# PyTorch Essentials: From Notebook to DGX Cluster

Hands-on refresher covering the PyTorch primitives you'll use throughout this notebook series.
Each section builds something real — from tensor ops through distributed training patterns.

Two modes throughout: **notebook mode** (quick experiments, single GPU) and **production mode**
(what changes when you're targeting hundreds of GPUs across multiple DGX nodes).

**Hardware:** CPU-only for Parts 1-5. GPU useful for Parts 6-7 but not required.
**Time:** ~3-4 hours working through all exercises.

In [9]:
# pip install torch torchvision lzma

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

ModuleNotFoundError: No module named '_lzma'

## Part 1: Tensor Foundations

Everything in PyTorch is a tensor. Before touching models or data pipelines, make sure
the fundamentals are second nature: creation, dtypes, device transfers, broadcasting, reshaping.

### Key concepts:

- **dtypes matter for performance.** `float32` is the default, `float16`/`bfloat16` for mixed precision,
  `long` (int64) for indices/labels. Using the wrong dtype silently kills performance or accuracy.
- **Device placement is explicit.** Tensors don't auto-migrate — you must `.to(device)` everything.
  Mismatched devices (CPU tensor + GPU tensor) = runtime error.
- **Broadcasting eliminates loops.** When shapes don't match, PyTorch auto-expands dimensions
  (right-aligned). This is how you apply per-channel normalization to a batch of images without a loop.
- **Views vs copies.** `.view()` and `.reshape()` share memory when possible (zero-cost).
  `.contiguous()` forces a copy when memory layout doesn't support the view. After `.transpose()`,
  the tensor is often non-contiguous — call `.contiguous()` before `.view()`.

**Your turn — four things to demonstrate:**

1. **Create tensors** with specific dtypes and devices
2. **Broadcasting** — normalize a (B, C, H, W) image batch with per-channel mean/std
3. **Reshaping** — go from (B, C, H, W) to (B, C, H*W) and back
4. **Indexing** — gather values from a 1D schedule using a batch of integer indices,
   then reshape for broadcasting (this is exactly what DDPM does)

**Look up:** `torch.randn`, `torch.zeros`, `torch.arange`, `torch.linspace`,
`Tensor.to()`, `Tensor.view()`, `Tensor.reshape()`, `Tensor.permute()`,
`Tensor.contiguous()`, `Tensor.unsqueeze()`

In [ ]:
# EXERCISE: Tensor fundamentals

# --- 1. Creation and dtypes ---
# YOUR CODE:
# a) Create a (2, 3, 64, 64) float32 tensor of random noise on `device`
images = ...

# b) Create a (2,) int64 tensor of random timesteps in [0, 1000) on `device`
#    Hint: torch.randint(low, high, size, device=...)
timesteps = ...

# c) Create a (1000,) float32 tensor: linearly spaced values from 1e-4 to 0.02, on `device`
betas = ...


# --- 2. Broadcasting: per-channel normalization ---
# Given a batch of images (B, C, H, W) with values in [0, 1],
# normalize to [-1, 1] using: (x - mean) / std, where mean=std=0.5 per channel.
#
# YOUR CODE:
# a) Create mean and std tensors of shape (1, 3, 1, 1) so they broadcast over B, H, W
#    Hint: torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1)
raw_images = torch.rand(4, 3, 64, 64, device=device)  # fake images in [0, 1]
mean = ...
std = ...
normalized = ...  # (raw_images - mean) / std


# --- 3. Reshaping: spatial flatten and unflatten ---
# YOUR CODE:
# a) Flatten spatial dims: (B, C, H, W) -> (B, C, H*W)
#    Hint: use .view(B, C, -1) or .flatten(2)
x = torch.randn(2, 256, 16, 16, device=device)
flat = ...

# b) Unflatten back: (B, C, H*W) -> (B, C, H, W)
#    Hint: use .view(B, C, 16, 16)
unflat = ...


# --- 4. Index + reshape for broadcasting (the DDPM pattern) ---
# This is the most important pattern in this section.
# You have a 1D schedule of 1000 values and a batch of timestep indices.
# You need to gather the values at those indices, then reshape so they
# can multiply a (B, C, H, W) tensor via broadcasting.
#
# YOUR CODE:
schedule = torch.linspace(0.9999, 0.01, 1000, device=device)  # fake alpha_bar
t = torch.tensor([0, 500, 999], device=device)  # 3 timesteps

# a) Gather values: schedule[t] — gives shape (3,)
values = ...

# b) Reshape to (3, 1, 1, 1) for broadcasting with (3, C, H, W) images
#    Hint: .view(-1, 1, 1, 1) or [:, None, None, None]
values_4d = ...

In [ ]:
# VERIFY: tensor operations

# 1. Creation
assert images.shape == (2, 3, 64, 64), f"images: expected (2, 3, 64, 64), got {images.shape}"
assert images.dtype == torch.float32, f"images dtype: expected float32, got {images.dtype}"
assert images.device.type == device.type, f"images device: expected {device.type}"
assert timesteps.shape == (2,), f"timesteps: expected (2,), got {timesteps.shape}"
assert timesteps.dtype == torch.int64, f"timesteps dtype: expected int64, got {timesteps.dtype}"
assert betas.shape == (1000,), f"betas: expected (1000,), got {betas.shape}"
print("1. Creation: OK")

# 2. Broadcasting
assert normalized.shape == (4, 3, 64, 64), f"normalized shape: {normalized.shape}"
assert normalized.min() >= -1.1 and normalized.max() <= 1.1, \
    f"normalized range: [{normalized.min():.2f}, {normalized.max():.2f}], expected ~[-1, 1]"
print(f"2. Broadcasting: OK — range [{normalized.min():.2f}, {normalized.max():.2f}]")

# 3. Reshaping
assert flat.shape == (2, 256, 256), f"flat: expected (2, 256, 256), got {flat.shape}"
assert unflat.shape == (2, 256, 16, 16), f"unflat: expected (2, 256, 16, 16), got {unflat.shape}"
assert torch.equal(x, unflat), "Round-trip reshape should be exact"
print("3. Reshaping: OK (round-trip exact)")

# 4. Index + reshape
assert values.shape == (3,), f"values: expected (3,), got {values.shape}"
assert values_4d.shape == (3, 1, 1, 1), f"values_4d: expected (3, 1, 1, 1), got {values_4d.shape}"
# Test that broadcasting works
test_imgs = torch.randn(3, 3, 64, 64, device=device)
scaled = values_4d * test_imgs  # should broadcast without error
assert scaled.shape == (3, 3, 64, 64), f"broadcast result: {scaled.shape}"
print("4. Index + broadcast: OK")
print(f"   schedule[0]={values[0]:.4f}, schedule[500]={values[1]:.4f}, schedule[999]={values[2]:.4f}")

## Part 2: Data Pipeline — Transforms, Datasets, DataLoaders

The data pipeline has three layers:

1. **Transforms** — preprocessing + augmentation applied per-sample
2. **Dataset** — maps an index to a (sample, label) tuple
3. **DataLoader** — batches, shuffles, and parallelizes data loading

### Notebook mode vs production mode:

| Concern | Notebook | Production (DGX cluster) |
|---------|----------|-------------------------|
| Dataset size | Fits in memory / on disk | Streamed from S3/GCS via WebDataset/MDS |
| Augmentation | Standard torchvision | Same transforms, but deterministic seeding per worker |
| DataLoader workers | 2-4 | 8-16 per GPU (tune to saturate the data bus) |
| Shuffling | In-memory shuffle | Shard-level shuffle + buffer shuffle (WebDataset) or deterministic shuffle (MDS) |
| Distributed | N/A | `DistributedSampler` ensures each GPU sees different data |
| Bottleneck check | `time.time()` around batch loading | `torch.profiler` or `nvidia-nsight` to find data stalls |

### Exercise A: Transforms

Build a training transform pipeline for 64x64 image generation. Training transforms
include augmentation (random horizontal flip); eval transforms do not.

**Look up:** `transforms.Compose`, `transforms.Resize`, `transforms.CenterCrop`,
`transforms.RandomHorizontalFlip`, `transforms.ToTensor`, `transforms.Normalize`

### Exercise B: Custom Dataset

Build a `SyntheticImageDataset` that generates random images on the fly.
This pattern is useful for testing pipelines without needing to download real data.

A Dataset needs two methods:
- `__len__` — returns number of samples
- `__getitem__` — takes an index, returns (sample, label)

**Look up:** `torch.utils.data.Dataset`, `__len__`, `__getitem__`

In [ ]:
# EXERCISE A: Build train and eval transform pipelines

from PIL import Image

IMG_SIZE = 64

# YOUR CODE:
# Training transform: RandomHorizontalFlip + Resize + CenterCrop + ToTensor + Normalize to [-1, 1]
# Why RandomHorizontalFlip for faces? Faces are roughly symmetric — this doubles effective
# dataset size for free. You would NOT use this for text images or anything with handedness.
train_transform = ...

# Eval transform: same but WITHOUT augmentation (no RandomHorizontalFlip)
eval_transform = ...


# EXERCISE B: Custom Dataset

class SyntheticImageDataset(Dataset):
    """Generates random RGB images. Useful for pipeline testing."""

    def __init__(self, num_samples: int = 1000, img_size: int = 64, num_classes: int = 10):
        # YOUR CODE: store params
        raise NotImplementedError

    def __len__(self) -> int:
        # YOUR CODE: return number of samples
        raise NotImplementedError

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        """Returns (image, label) where image is (C, H, W) in [-1, 1]."""
        # YOUR CODE:
        # 1. Use a per-index seed for reproducibility: torch.manual_seed(idx)
        #    (so dataset[42] always returns the same image)
        # 2. Generate random image: torch.randn(3, img_size, img_size)
        # 3. Generate label: idx % num_classes
        # 4. Return (image, label)
        raise NotImplementedError


dataset = SyntheticImageDataset(num_samples=2000, img_size=IMG_SIZE)
print(f"Dataset size: {len(dataset)}")
print(f"Sample shape: {dataset[0][0].shape}, label: {dataset[0][1]}")

In [ ]:
# VERIFY: transforms and dataset

# Test transforms with a synthetic PIL image
test_pil = Image.fromarray(np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8))

train_out = train_transform(test_pil)
eval_out = eval_transform(test_pil)
assert train_out.shape == (3, IMG_SIZE, IMG_SIZE), f"Train transform shape: {train_out.shape}"
assert eval_out.shape == (3, IMG_SIZE, IMG_SIZE), f"Eval transform shape: {eval_out.shape}"
assert train_out.min() >= -1.1 and train_out.max() <= 1.1, "Train values should be in [-1, 1]"
print(f"Train transform: {test_pil.size} PIL -> {train_out.shape} tensor, range [{train_out.min():.2f}, {train_out.max():.2f}]")
print(f"Eval transform:  {test_pil.size} PIL -> {eval_out.shape} tensor, range [{eval_out.min():.2f}, {eval_out.max():.2f}]")

# Test dataset
assert len(dataset) == 2000
img, label = dataset[0]
assert img.shape == (3, IMG_SIZE, IMG_SIZE), f"Dataset sample shape: {img.shape}"
assert isinstance(label, int), f"Label should be int, got {type(label)}"

# Reproducibility: same index should return same image
img_a = dataset[42][0]
img_b = dataset[42][0]
assert torch.equal(img_a, img_b), "Same index should return identical data"
print(f"Dataset: {len(dataset)} samples, shape {img.shape}, labels 0-{dataset.num_classes - 1}")
print("Reproducibility: OK")

### Part 2b: DataLoader Mechanics

The DataLoader wraps a Dataset and handles batching, shuffling, and parallel loading.

**Key parameters:**

| Parameter | What it does | Notebook | Production |
|-----------|-------------|----------|------------|
| `batch_size` | Samples per batch | 32-128 | 32-128 per GPU (scale with grad accumulation) |
| `shuffle` | Randomize order each epoch | `True` | `False` (let `DistributedSampler` handle it) |
| `num_workers` | Parallel data loading processes | 2-4 | 8-16 (tune until GPU is saturated) |
| `pin_memory` | Pre-stage batches in GPU-ready memory | `True` if GPU | `True` always |
| `drop_last` | Drop incomplete final batch | Optional | `True` (uneven batches cause DDP sync issues) |
| `persistent_workers` | Keep worker processes alive between epochs | `False` | `True` (avoids respawn overhead) |
| `prefetch_factor` | Batches pre-loaded per worker | 2 (default) | 2-4 (higher if data loading is bottleneck) |

**Your turn:**

1. Create a standard DataLoader for notebook experimentation
2. Create a production-style DataLoader with all the performance flags
3. Write a custom `collate_fn` — needed when your samples aren't uniform shape

**Look up:** `DataLoader`, `collate_fn`, `pin_memory`, `persistent_workers`

In [ ]:
# EXERCISE: DataLoader configurations

BATCH_SIZE = 32

# YOUR CODE:
# 1. Notebook-style DataLoader: simple, shuffle=True, 2 workers
notebook_loader = ...

# 2. Production-style DataLoader: all performance flags enabled
#    shuffle=True, num_workers=4, pin_memory=True, drop_last=True,
#    persistent_workers=True, prefetch_factor=2
#    Note: persistent_workers requires num_workers > 0
production_loader = ...


# 3. Custom collate_fn — for variable-length data
# Scenario: your dataset returns captions of different lengths alongside images.
# Default collate can't stack variable-length tensors. You need a custom collate_fn.

class ImageCaptionDataset(Dataset):
    """Returns (image, caption_tokens) where caption length varies."""

    def __init__(self, num_samples: int = 500):
        self.num_samples = num_samples

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        torch.manual_seed(idx)
        image = torch.randn(3, 64, 64)
        caption_len = torch.randint(5, 30, (1,)).item()  # variable length!
        caption = torch.randint(0, 10000, (caption_len,))  # fake token IDs
        return image, caption


def caption_collate_fn(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> dict:
    """Collate images (stack) and captions (pad to max length in batch).

    Args:
        batch: list of (image, caption) tuples from the dataset

    Returns:
        dict with keys:
        - 'images': (B, C, H, W) tensor
        - 'captions': (B, max_len) tensor, padded with 0s
        - 'caption_lengths': (B,) tensor of original lengths
    """
    # YOUR CODE:
    # 1. Separate images and captions from the batch
    #    images, captions = zip(*batch)
    # 2. Stack images: torch.stack(images)
    # 3. Find max caption length in this batch
    # 4. Pad each caption to max_len with zeros using F.pad or manual approach
    #    Hint: F.pad(caption, (0, max_len - len(caption)), value=0)
    # 5. Stack padded captions
    # 6. Record original lengths
    raise NotImplementedError


caption_dataset = ImageCaptionDataset(500)
caption_loader = DataLoader(caption_dataset, batch_size=8, collate_fn=caption_collate_fn)
sample_batch = next(iter(caption_loader))
print(f"Images: {sample_batch['images'].shape}")
print(f"Captions: {sample_batch['captions'].shape}")
print(f"Lengths: {sample_batch['caption_lengths']}")

In [ ]:
# VERIFY: DataLoaders

# Notebook loader
batch = next(iter(notebook_loader))
imgs, labels = batch
assert imgs.shape == (BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE), f"Batch shape: {imgs.shape}"
print(f"Notebook loader: batch {imgs.shape}, {len(notebook_loader)} batches/epoch")

# Production loader
batch = next(iter(production_loader))
imgs, labels = batch
assert imgs.shape == (BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE), f"Batch shape: {imgs.shape}"
print(f"Production loader: batch {imgs.shape}, {len(production_loader)} batches/epoch (drop_last=True)")

# Caption loader with custom collate
batch = sample_batch
assert batch["images"].shape[0] == 8, f"Batch size: {batch['images'].shape[0]}"
assert batch["captions"].shape[0] == 8, f"Captions batch: {batch['captions'].shape[0]}"
assert batch["captions"].shape[1] <= 30, "Max caption length should be <= 30"
assert len(batch["caption_lengths"]) == 8, "Should have 8 lengths"
# Check padding: each caption should have zeros after its actual length
for i in range(8):
    L = batch["caption_lengths"][i].item()
    if L < batch["captions"].shape[1]:
        assert (batch["captions"][i, L:] == 0).all(), f"Sample {i}: padding should be zeros"
print(f"Custom collate: images {batch['images'].shape}, captions {batch['captions'].shape}")
print("All DataLoaders working correctly!")

## Part 3: nn.Module — Custom Layers & Parameter Management

Every model in PyTorch is built from `nn.Module` subclasses. Understanding the mechanics —
parameters, buffers, hooks, state_dict — is essential for both building and debugging.

### Parameters vs Buffers:

- **Parameters** (`nn.Parameter`) — tensors that get gradients and are updated by the optimizer.
  Example: conv weights, linear weights, learned embeddings.
- **Buffers** (`register_buffer`) — tensors that are part of the module state (saved/loaded
  with the model, moved with `.to(device)`) but do NOT get gradients.
  Example: running mean in BatchNorm, the noise schedule in DDPM.

**Why buffers matter for diffusion models:** The alpha_bar schedule is a fixed tensor that
must live on the same device as the model. If you store it as a plain attribute, `.to(device)`
won't move it. Use `register_buffer` and it auto-migrates.

### Your turn — build a `ConditionalResBlock`:

A residual block that takes a feature map AND a conditioning vector (like a timestep embedding).
This is the core building block in diffusion model U-Nets.

Architecture:
```
x (B, in_ch, H, W)
  → Conv2d(in_ch, out_ch, 3, padding=1) → GroupNorm → SiLU
  → ADD conditioning projection: Linear(cond_dim → out_ch), reshaped to (B, out_ch, 1, 1)
  → Conv2d(out_ch, out_ch, 3, padding=1) → GroupNorm → SiLU
  → ADD residual (with 1x1 conv if in_ch != out_ch)
```

Also register a buffer: a counter that tracks how many forward passes have been run
(useful for debugging/profiling).

**Look up:** `nn.Module`, `nn.Parameter`, `register_buffer`, `nn.Conv2d`,
`nn.GroupNorm`, `nn.Linear`, `nn.Identity`, `F.silu`

In [ ]:
# EXERCISE: Build a ConditionalResBlock

class ConditionalResBlock(nn.Module):
    """Residual block with conditioning injection (e.g., timestep embedding)."""

    def __init__(self, in_ch: int, out_ch: int, cond_dim: int):
        super().__init__()
        # YOUR CODE: define all layers
        # - conv1: Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        # - norm1: GroupNorm(8, out_ch)  — 8 groups is standard
        # - conv2: Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        # - norm2: GroupNorm(8, out_ch)
        # - cond_proj: Linear(cond_dim, out_ch)  — projects conditioning to channel dim
        # - residual: Conv2d(in_ch, out_ch, 1) if in_ch != out_ch, else nn.Identity()
        #
        # Register a buffer:
        # - self.register_buffer("forward_count", torch.tensor(0, dtype=torch.long))

        raise NotImplementedError

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        """Args: x (B, in_ch, H, W), cond (B, cond_dim). Returns: (B, out_ch, H, W)."""
        # YOUR CODE:
        # 1. h = F.silu(self.norm1(self.conv1(x)))
        # 2. c = F.silu(self.cond_proj(cond))[:, :, None, None]  — reshape to (B, out_ch, 1, 1)
        # 3. h = h + c   — add conditioning
        # 4. h = F.silu(self.norm2(self.conv2(h)))
        # 5. Increment forward_count: self.forward_count += 1
        # 6. return h + self.residual(x)

        raise NotImplementedError

In [ ]:
# VERIFY: ConditionalResBlock

# Test with channel change: 64 -> 128
block = ConditionalResBlock(in_ch=64, out_ch=128, cond_dim=256).to(device)

x = torch.randn(2, 64, 16, 16, device=device)
cond = torch.randn(2, 256, device=device)
out = block(x, cond)

assert out.shape == (2, 128, 16, 16), f"Output shape: {out.shape}"
print(f"Forward pass: (2, 64, 16, 16) + cond (2, 256) -> {out.shape}")

# Test without channel change: 128 -> 128 (residual should use Identity)
block_same = ConditionalResBlock(in_ch=128, out_ch=128, cond_dim=256).to(device)
x2 = torch.randn(2, 128, 16, 16, device=device)
out2 = block_same(x2, cond)
assert out2.shape == (2, 128, 16, 16), f"Same-channel output: {out2.shape}"
print(f"Same channels: (2, 128, 16, 16) -> {out2.shape}")

# Check parameter counts
params = sum(p.numel() for p in block.parameters())
print(f"Parameters (64->128): {params:,}")

# Check buffer exists and increments
assert hasattr(block, "forward_count"), "Missing forward_count buffer"
assert block.forward_count.item() == 1, f"forward_count should be 1, got {block.forward_count.item()}"
_ = block(x, cond)  # run again
assert block.forward_count.item() == 2, f"forward_count should be 2, got {block.forward_count.item()}"
print(f"Buffer tracking: forward_count = {block.forward_count.item()}")

# Check that buffer moves with .to()
block_cpu = block.to("cpu")
assert block_cpu.forward_count.device.type == "cpu", "Buffer should move with .to()"
print("Buffer device migration: OK")

# Check state_dict includes buffer
sd = block.state_dict()
assert "forward_count" in sd, "Buffer should appear in state_dict"
print(f"state_dict keys: {list(sd.keys())}")

# Gradient check: backward should work
block = block.to(device)
out = block(x, cond)
out.sum().backward()
assert block.conv1.weight.grad is not None, "conv1 should have gradients"
print("Backward pass: OK")

### Part 3b: Hooks — Inspecting What's Happening Inside

Hooks let you intercept data flowing through a module without modifying the code.
Essential for debugging, visualization, and feature extraction.

Three kinds:
- **Forward hook** — runs after `forward()`, gets (module, input, output). Use for:
  feature visualization, activation statistics, intermediate outputs.
- **Backward hook** — runs during `backward()`, gets gradient information. Use for:
  gradient monitoring, detecting vanishing/exploding gradients.
- **Forward pre-hook** — runs before `forward()`. Use for: input modification, shape logging.

**Production use case:** During large-scale training, you often register hooks to monitor
gradient norms per layer, activation magnitudes, or NaN detection — without modifying model code.

**Your turn:**

1. Register a forward hook that captures activation statistics (mean, std, min, max)
2. Register a backward hook that captures gradient norms
3. Run a forward + backward pass and inspect the captured stats

**Look up:** `module.register_forward_hook`, `module.register_full_backward_hook`

In [ ]:
# EXERCISE: Hooks for monitoring activations and gradients

# Storage for hook outputs
activation_stats = {}
gradient_stats = {}


def activation_hook(name: str):
    """Returns a forward hook that logs activation stats for a named layer."""
    # YOUR CODE:
    # Define an inner function: hook(module, input, output)
    # Store in activation_stats[name]: dict with mean, std, min, max of output
    # Hint: use output.detach() to avoid holding onto the computation graph
    def hook(module, input, output):
        raise NotImplementedError
    return hook


def gradient_hook(name: str):
    """Returns a backward hook that logs gradient norms for a named layer."""
    # YOUR CODE:
    # Define an inner function: hook(module, grad_input, grad_output)
    # grad_output is a tuple of tensors (one per output)
    # Store in gradient_stats[name]: the L2 norm of grad_output[0]
    # Hint: grad_output[0].detach().norm().item()
    def hook(module, grad_input, grad_output):
        raise NotImplementedError
    return hook


# Build a small model and register hooks on every conv layer
model = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(64, 10),
).to(device)

# YOUR CODE: register hooks
# Register activation_hook on model[0] (first conv) with name "conv1"
# Register activation_hook on model[2] (second conv) with name "conv2"
# Register gradient_hook on model[0] with name "conv1"
# Register gradient_hook on model[2] with name "conv2"
#
# Hint: handle = model[0].register_forward_hook(activation_hook("conv1"))
#        handle = model[0].register_full_backward_hook(gradient_hook("conv1"))
# (Store handles if you want to remove hooks later with handle.remove())

handles = []
# YOUR CODE: append handles
raise NotImplementedError

# Run forward + backward
x = torch.randn(4, 3, 32, 32, device=device)
out = model(x)
loss = out.sum()
loss.backward()

# Print captured stats
print("Activation stats:")
for name, stats in activation_stats.items():
    print(f"  {name}: mean={stats['mean']:.4f}, std={stats['std']:.4f}, "
          f"min={stats['min']:.4f}, max={stats['max']:.4f}")

print("\nGradient norms:")
for name, norm in gradient_stats.items():
    print(f"  {name}: grad_norm={norm:.4f}")

# Clean up hooks
for h in handles:
    h.remove()

In [ ]:
# VERIFY: hooks captured data

assert "conv1" in activation_stats, "Missing conv1 activation stats"
assert "conv2" in activation_stats, "Missing conv2 activation stats"
assert "conv1" in gradient_stats, "Missing conv1 gradient stats"
assert "conv2" in gradient_stats, "Missing conv2 gradient stats"

for name in ["conv1", "conv2"]:
    stats = activation_stats[name]
    assert all(k in stats for k in ["mean", "std", "min", "max"]), f"{name} missing stats keys"
    assert isinstance(gradient_stats[name], float), f"{name} grad norm should be float"

# After .remove(), hooks should no longer fire
activation_stats.clear()
gradient_stats.clear()
model.zero_grad()
out = model(x)
out.sum().backward()
assert len(activation_stats) == 0, "Hooks should be removed"
assert len(gradient_stats) == 0, "Hooks should be removed"
print("Hooks registered, captured, and cleaned up correctly!")

## Part 4: Custom Loss Functions

PyTorch's built-in losses (`F.mse_loss`, `F.cross_entropy`, `F.l1_loss`) cover basics.
But real training often uses **multi-component losses** — a weighted combination of terms
that each push the model in a different direction.

**Examples from diffusion/generative models:**

| Loss term | What it does | Used in |
|-----------|-------------|--------|
| MSE / L2 | Pixel-level reconstruction | DDPM noise prediction |
| L1 | Sharper reconstruction (less blur than L2) | Image super-resolution |
| Perceptual (LPIPS) | Feature-level similarity via pretrained net | VAE decoder training |
| Adversarial | "Fool the discriminator" | GAN-based refinement |
| KL divergence | Regularize latent distribution | VAE encoder |
| CLIP similarity | Text-image alignment | Guided diffusion |

### Your turn:

Build a `CombinedLoss` module that computes a weighted sum of:
1. **MSE loss** (reconstruction)
2. **L1 loss** (sharpness)
3. **Frequency loss** — penalizes differences in the frequency domain (DCT or FFT).
   This helps preserve high-frequency details that MSE tends to blur.

The loss should return both the total and a breakdown dict (for logging).

**Look up:** `F.mse_loss`, `F.l1_loss`, `torch.fft.rfft2` (2D real FFT)

In [ ]:
# EXERCISE: Multi-component loss function

class CombinedLoss(nn.Module):
    """Weighted combination of MSE, L1, and frequency-domain losses."""

    def __init__(self, w_mse: float = 1.0, w_l1: float = 0.1, w_freq: float = 0.05):
        super().__init__()
        # YOUR CODE:
        # Store weights. Use register_buffer so they appear in state_dict
        # and move with .to(device).
        # Hint: self.register_buffer("w_mse", torch.tensor(w_mse))
        raise NotImplementedError

    def frequency_loss(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """L1 loss in frequency domain (2D FFT magnitude)."""
        # YOUR CODE:
        # 1. Compute 2D FFT of pred and target: torch.fft.rfft2(tensor)
        # 2. Take magnitude: .abs()
        # 3. Return F.l1_loss between the magnitudes
        raise NotImplementedError

    def forward(
        self, pred: torch.Tensor, target: torch.Tensor
    ) -> tuple[torch.Tensor, dict[str, float]]:
        """Compute weighted total loss + per-component breakdown.

        Returns:
            total: scalar loss tensor (for backward)
            breakdown: dict of {name: value} for logging
        """
        # YOUR CODE:
        # 1. mse = F.mse_loss(pred, target)
        # 2. l1 = F.l1_loss(pred, target)
        # 3. freq = self.frequency_loss(pred, target)
        # 4. total = w_mse * mse + w_l1 * l1 + w_freq * freq
        # 5. Return (total, {"mse": mse.item(), "l1": l1.item(), "freq": freq.item(), "total": total.item()})
        raise NotImplementedError

In [ ]:
# VERIFY: combined loss

loss_fn = CombinedLoss(w_mse=1.0, w_l1=0.1, w_freq=0.05).to(device)

pred = torch.randn(4, 3, 64, 64, device=device, requires_grad=True)
target = torch.randn(4, 3, 64, 64, device=device)

total, breakdown = loss_fn(pred, target)

assert total.shape == torch.Size([]), f"Total loss should be scalar, got {total.shape}"
assert total.requires_grad, "Total loss should have grad"
assert all(k in breakdown for k in ["mse", "l1", "freq", "total"]), f"Missing keys: {breakdown.keys()}"

# Backward should work
total.backward()
assert pred.grad is not None, "Gradients should flow through"

print("Loss breakdown:")
for k, v in breakdown.items():
    print(f"  {k}: {v:.4f}")

# Verify weights affect the total correctly
expected_total = breakdown["mse"] * 1.0 + breakdown["l1"] * 0.1 + breakdown["freq"] * 0.05
assert abs(breakdown["total"] - expected_total) < 1e-3, \
    f"Total {breakdown['total']:.4f} != weighted sum {expected_total:.4f}"
print(f"\nWeighted sum verified: {breakdown['total']:.4f}")
print("Combined loss: OK!")

## Part 5: The Complete Training Loop

A real training loop has more moving parts than `loss.backward(); optimizer.step()`.
Here's everything a production loop handles:

```
for epoch:
  for batch:
    1. Forward pass
    2. Compute loss
    3. Backward pass
    4. Gradient clipping  ← prevents training explosions
    5. Optimizer step
    6. LR scheduler step  ← usually per-batch (not per-epoch) for modern schedulers
    7. Logging
  end batch
  Checkpoint save
  Validation
end epoch
```

### Gradient clipping:
Caps gradient magnitudes to prevent a single bad batch from destroying the model.
`torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`

### LR schedulers:
- **CosineAnnealingLR** — smooth decay. Good default.
- **CosineAnnealingWarmRestarts** — periodic restarts. Helps escape local minima.
- **OneCycleLR** — warmup + cosine decay in one shot. Often best for fixed-budget training.
- **Linear warmup** — ramp LR from 0 to target over first N steps. Critical for transformers/diffusion.

### Checkpointing:
Save everything needed to resume: model state, optimizer state, scheduler state, epoch, step, loss.

**Your turn:** Implement a complete training loop that trains a small CNN on synthetic data.
Include: optimizer, cosine LR scheduler with linear warmup, gradient clipping, checkpoint
save/load with full resume capability.

**Look up:** `torch.optim.AdamW`, `torch.optim.lr_scheduler.CosineAnnealingLR`,
`torch.nn.utils.clip_grad_norm_`, `torch.save`, `torch.load`

In [ ]:
# EXERCISE: Complete training loop with scheduler, clipping, checkpointing

# Simple CNN for the exercise
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x).flatten(1))


# Config
EPOCHS = 5
LR = 1e-3
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
CKPT_PATH = Path("./checkpoints/training_demo.pt")
CKPT_PATH.parent.mkdir(exist_ok=True)

# Setup
model = SimpleCNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# Scheduler: cosine annealing (we'll add warmup manually)
total_steps = EPOCHS * len(notebook_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# Training state
start_epoch = 0
global_step = 0
train_losses = []


# YOUR CODE: implement save_checkpoint and load_checkpoint

def save_checkpoint(path: Path, epoch: int, step: int, losses: list[float]) -> None:
    """Save full training state for resumability."""
    # YOUR CODE:
    # torch.save a dict containing:
    # - "model": model.state_dict()
    # - "optimizer": optimizer.state_dict()
    # - "scheduler": scheduler.state_dict()
    # - "epoch": epoch
    # - "global_step": step
    # - "losses": losses
    raise NotImplementedError


def load_checkpoint(path: Path) -> tuple[int, int, list[float]]:
    """Load training state. Returns (start_epoch, global_step, losses)."""
    # YOUR CODE:
    # 1. ckpt = torch.load(path, map_location=device, weights_only=False)
    #    (weights_only=False because we stored non-tensor data)
    # 2. model.load_state_dict(ckpt["model"])
    # 3. optimizer.load_state_dict(ckpt["optimizer"])
    # 4. scheduler.load_state_dict(ckpt["scheduler"])
    # 5. Return (ckpt["epoch"], ckpt["global_step"], ckpt["losses"])
    raise NotImplementedError


# YOUR CODE: implement the training loop

model.train()
for epoch in range(start_epoch, EPOCHS):
    epoch_losses = []

    for batch_imgs, batch_labels in notebook_loader:
        batch_imgs = batch_imgs.to(device)
        batch_labels = batch_labels.to(device)

        # YOUR CODE:
        # 1. Linear warmup: if global_step < WARMUP_STEPS, scale LR
        #    warmup_factor = min(1.0, global_step / WARMUP_STEPS)
        #    for pg in optimizer.param_groups: pg["lr"] = LR * warmup_factor
        #    (Only apply warmup override during warmup period)
        #
        # 2. Forward pass: logits = model(batch_imgs)
        # 3. Loss: F.cross_entropy(logits, batch_labels)
        # 4. Backward: optimizer.zero_grad(), loss.backward()
        # 5. Gradient clipping: torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        # 6. Step: optimizer.step(), scheduler.step()
        # 7. Track: append loss.item() to epoch_losses, increment global_step

        raise NotImplementedError

    avg_loss = np.mean(epoch_losses)
    train_losses.append(avg_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch + 1}/{EPOCHS} — loss: {avg_loss:.4f}, lr: {current_lr:.6f}")

# Save final checkpoint
save_checkpoint(CKPT_PATH, EPOCHS, global_step, train_losses)
print(f"\nCheckpoint saved to {CKPT_PATH}")

In [ ]:
# VERIFY: training loop and checkpointing

assert len(train_losses) == EPOCHS, f"Expected {EPOCHS} loss values, got {len(train_losses)}"
assert all(isinstance(l, (float, np.floating)) for l in train_losses), "Losses should be floats"
print(f"Training completed: {EPOCHS} epochs, {global_step} total steps")

# Plot training loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, marker="o")
ax.set(xlabel="Epoch", ylabel="Loss", title="Training Loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Test checkpoint resume: load into fresh model, verify state matches
assert CKPT_PATH.exists(), f"Checkpoint not found at {CKPT_PATH}"
model_fresh = SimpleCNN().to(device)
opt_fresh = torch.optim.AdamW(model_fresh.parameters(), lr=LR)
sched_fresh = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fresh, T_max=total_steps)

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model_fresh.load_state_dict(ckpt["model"])
opt_fresh.load_state_dict(ckpt["optimizer"])
sched_fresh.load_state_dict(ckpt["scheduler"])

# Verify loaded model produces same output
model.eval()
model_fresh.eval()
with torch.no_grad():
    test_input = torch.randn(1, 3, 64, 64, device=device)
    orig_out = model(test_input)
    loaded_out = model_fresh(test_input)
    assert torch.allclose(orig_out, loaded_out, atol=1e-6), "Loaded model output doesn't match!"

print(f"Checkpoint verified: epoch={ckpt['epoch']}, step={ckpt['global_step']}")
print("Resume capability: OK")

## Part 6: Mixed Precision & Gradient Accumulation

Two techniques that let you train bigger models and use larger effective batch sizes
without buying more hardware.

### Mixed Precision (AMP)

Run forward pass in float16/bfloat16 (faster, less memory) and backward pass in float32
(for numerical stability). On modern GPUs this is essentially free performance.

```python
scaler = torch.amp.GradScaler()                        # handles float16 underflow
with torch.amp.autocast(device_type="cuda"):            # forward in fp16
    loss = model(x)
scaler.scale(loss).backward()                           # scale to prevent underflow
scaler.step(optimizer)                                  # unscale + step
scaler.update()                                         # adjust scale factor
```

**bfloat16 vs float16:** bfloat16 has same range as float32 (just less precision), so it
rarely needs loss scaling. Preferred on A100/H100. float16 has limited range, needs GradScaler.

### Gradient Accumulation

Simulate larger batch sizes by accumulating gradients over N mini-batches before stepping.
Effective batch size = `batch_size * accumulation_steps`.

Why? On a DGX cluster with 256 GPUs, your effective batch per GPU might be 32,
but you want an effective global batch of 2048. That's 32 * 256 / 32 = natural with DDP.
But on a single RTX 2080, you can only fit batch_size=16 in memory. With 4 accumulation
steps, you simulate batch_size=64.

**Key detail:** divide the loss by `accumulation_steps` so gradients scale correctly.
Call `optimizer.zero_grad()` only after the accumulation window.

**Your turn:** Implement a training step with both AMP and gradient accumulation.

**Look up:** `torch.amp.autocast`, `torch.amp.GradScaler`, `scaler.scale`,
`scaler.step`, `scaler.update`

In [ ]:
# EXERCISE: Training with AMP + gradient accumulation

ACCUMULATION_STEPS = 4  # effective batch = BATCH_SIZE * 4 = 128
AMP_ENABLED = device.type == "cuda"  # AMP only meaningful on GPU
AMP_DTYPE = torch.bfloat16 if (AMP_ENABLED and torch.cuda.is_bf16_supported()) else torch.float16

model = SimpleCNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler(enabled=AMP_ENABLED)

model.train()
losses = []

# YOUR CODE: Training loop with AMP + gradient accumulation
# Run for 2 epochs

for epoch in range(2):
    epoch_losses = []

    for step, (imgs, labels) in enumerate(notebook_loader):
        imgs = imgs.to(device)
        labels = labels.to(device)

        # YOUR CODE:
        # 1. AMP forward pass:
        #    with torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
        #        logits = model(imgs)
        #        loss = F.cross_entropy(logits, labels)
        #        loss = loss / ACCUMULATION_STEPS  # scale for accumulation
        #
        # 2. Scaled backward:
        #    scaler.scale(loss).backward()
        #
        # 3. Only step every ACCUMULATION_STEPS:
        #    if (step + 1) % ACCUMULATION_STEPS == 0:
        #        scaler.unscale_(optimizer)  # unscale before clipping
        #        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        #        scaler.step(optimizer)
        #        scaler.update()
        #        optimizer.zero_grad()
        #
        # 4. Track loss (multiply back by ACCUMULATION_STEPS for true loss value)

        raise NotImplementedError

    avg_loss = np.mean(epoch_losses) if epoch_losses else 0.0
    losses.append(avg_loss)
    print(f"Epoch {epoch + 1}/2 — loss: {avg_loss:.4f}")

print(f"\nEffective batch size: {BATCH_SIZE} x {ACCUMULATION_STEPS} = {BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"AMP: {'enabled (' + str(AMP_DTYPE) + ')' if AMP_ENABLED else 'disabled (CPU)'}")

In [ ]:
# VERIFY: AMP + gradient accumulation

assert len(losses) == 2, f"Expected 2 epoch losses, got {len(losses)}"
assert all(l > 0 for l in losses), "Losses should be positive"
print(f"Training completed: losses = {[f'{l:.4f}' for l in losses]}")

# Verify model still works in eval mode
model.eval()
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
        test_out = model(torch.randn(1, 3, 64, 64, device=device))
        assert test_out.shape == (1, 10), f"Eval output shape: {test_out.shape}"

print(f"Grad accumulation: {ACCUMULATION_STEPS} steps")
print(f"Effective batch: {BATCH_SIZE * ACCUMULATION_STEPS}")
print("AMP + gradient accumulation: OK!")

## Part 7: Distributed Training — From Single GPU to DGX Cluster

This section covers the patterns for scaling from your RTX 2080 to hundreds of GPUs.
The code here is **read-and-understand** — you can't meaningfully run DDP on a single GPU,
but you need to know the patterns cold because they show up in every large-scale codebase.

### The distributed training mental model:

```
DGX Node 0                    DGX Node 1
┌─────────────────────┐      ┌─────────────────────┐
│ GPU 0  GPU 1        │      │ GPU 0  GPU 1        │
│ rank=0 rank=1       │      │ rank=2 rank=3       │
│ batch₀ batch₁       │      │ batch₂ batch₃       │
│   ↓      ↓          │      │   ↓      ↓          │
│ grad₀  grad₁        │      │ grad₂  grad₃        │
└────────┬────────────┘      └────────┬────────────┘
         └──── AllReduce(mean) ───────┘
                    ↓
         All GPUs get averaged grads
         All GPUs do same optimizer step
         All GPUs stay in sync
```

### Three levels of parallelism:

| Strategy | What it does | When to use |
|----------|-------------|-------------|
| **DataParallel (DP)** | Replicates model per batch split. Single-process. | Never in production. Legacy. |
| **DistributedDataParallel (DDP)** | One process per GPU, AllReduce gradients. | Default for multi-GPU. |
| **FSDP (Fully Sharded)** | Shards model parameters + gradients + optimizer across GPUs. | When model doesn't fit in one GPU. |

### DDP essentials:

```python
# Each process runs this script independently
# rank = this GPU's global ID (0..world_size-1)
# local_rank = this GPU's ID within this node (0..N_gpus-1)
# world_size = total number of GPUs across all nodes

torch.distributed.init_process_group(backend="nccl")  # NCCL for GPU
torch.cuda.set_device(local_rank)

model = MyModel().to(local_rank)
model = DDP(model, device_ids=[local_rank])

# DistributedSampler ensures each GPU gets different data
sampler = DistributedSampler(dataset, shuffle=True)
loader = DataLoader(dataset, sampler=sampler, batch_size=32)  # NO shuffle=True with sampler

for epoch in range(epochs):
    sampler.set_epoch(epoch)  # CRITICAL: changes shuffling each epoch
    for batch in loader:
        # ... normal training loop — DDP handles gradient sync automatically
```

### What changes at DGX scale:

| Concern | Single GPU | DGX Cluster (256 GPUs) |
|---------|-----------|------------------------|
| Launch | `python train.py` | `torchrun --nproc_per_node=8 --nnodes=32 --rdzv_backend=c10d ...` |
| Data loading | `DataLoader(shuffle=True)` | `DistributedSampler` + `set_epoch()` |
| Model wrapping | None | `DDP(model)` or `FSDP(model)` |
| Batch normalization | `nn.BatchNorm2d` | `nn.SyncBatchNorm.convert_sync_batchnorm(model)` |
| Logging | Every step | Only on rank 0 (`if rank == 0: log(...)`) |
| Checkpointing | `torch.save(model.state_dict())` | `model.module.state_dict()` (unwrap DDP) |
| Gradient accumulation | Divide by accum_steps | Also divide by accum_steps; DDP syncs after `.backward()` |
| Communication overhead | None | AllReduce time scales with model size. Overlap with compute. |

In [ ]:
# REFERENCE: DDP training script pattern
# This is the canonical structure. Read it, understand each line.
# You'd run this with: torchrun --nproc_per_node=8 train_ddp.py

DDP_SCRIPT = """
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler


def setup():
    # torchrun sets these env vars automatically
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = dist.get_world_size()
    torch.cuda.set_device(local_rank)
    return rank, local_rank, world_size


def cleanup():
    dist.destroy_process_group()


def train():
    rank, local_rank, world_size = setup()
    device = torch.device(f"cuda:{local_rank}")

    # Model — wrap with DDP
    model = MyModel().to(device)
    model = nn.SyncBatchNorm.convert_sync_batchnorm(model)  # if using BatchNorm
    model = DDP(model, device_ids=[local_rank])

    # Data — DistributedSampler splits data across GPUs
    dataset = MyDataset()
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(
        dataset,
        batch_size=32,          # per-GPU batch size
        sampler=sampler,        # replaces shuffle=True
        num_workers=8,
        pin_memory=True,
        drop_last=True,         # avoid uneven batches across GPUs
        persistent_workers=True,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4 * world_size)  # scale LR
    scaler = torch.amp.GradScaler()

    for epoch in range(100):
        sampler.set_epoch(epoch)  # CRITICAL: ensures different shuffle each epoch
        model.train()

        for batch in loader:
            imgs = batch["images"].to(device)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = compute_loss(model, imgs)

            optimizer.zero_grad()
            scaler.scale(loss).backward()       # DDP syncs gradients in backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

        # Only save on rank 0 — all ranks have identical weights after AllReduce
        if rank == 0:
            torch.save({
                "model": model.module.state_dict(),   # .module unwraps DDP
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            }, f"checkpoint_epoch{epoch}.pt")

        # Barrier: all ranks wait here before next epoch
        dist.barrier()

    cleanup()


if __name__ == "__main__":
    train()
"""

print("DDP script pattern loaded (reference only — not executed)")
print("Launch with: torchrun --nproc_per_node=8 train_ddp.py")
print("Multi-node:  torchrun --nproc_per_node=8 --nnodes=32 --rdzv_backend=c10d "
      "--rdzv_endpoint=<master_ip>:29500 train_ddp.py")

In [ ]:
# REFERENCE: FSDP pattern — for models that don't fit on a single GPU
#
# FSDP shards parameters, gradients, AND optimizer state across GPUs.
# Each GPU only holds 1/N of the model. During forward, parameters are
# all-gathered just-in-time. After backward, they're resharded.
#
# When to use FSDP vs DDP:
# - DDP: model fits on one GPU. Simpler. Less communication overhead.
# - FSDP: model is too large for one GPU's memory. Trades communication for memory.
#
# Example: A 7B parameter model in float32 = 28 GB. Doesn't fit on a 24GB GPU.
# With FSDP across 4 GPUs: each holds ~7 GB of params + grads + optimizer.

FSDP_SNIPPET = """
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

# Define what layers to shard individually
# Each TransformerBlock becomes a separately sharded FSDP unit
auto_wrap_policy = transformer_auto_wrap_policy(
    transformer_layer_cls={TransformerBlock},
)

# Mixed precision config for FSDP
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,     # store params in bf16
    reduce_dtype=torch.bfloat16,    # AllReduce in bf16
    buffer_dtype=torch.bfloat16,    # buffers in bf16
)

model = FSDP(
    model,
    auto_wrap_policy=auto_wrap_policy,
    mixed_precision=mp_policy,
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # shard everything
    device_id=local_rank,
)

# Training loop is identical to DDP — FSDP handles the rest
"""

print("FSDP pattern loaded (reference only)")
print("\nSharding strategies:")
print("  FULL_SHARD  — shard params + grads + optimizer (most memory efficient)")
print("  SHARD_GRAD_OP — shard grads + optimizer only (faster, less memory savings)")
print("  NO_SHARD    — equivalent to DDP (for comparison/debugging)")

### Part 7b: Practical Exercise — DistributedSampler Simulation

You can't run DDP on one GPU, but you CAN verify you understand DistributedSampler.
Simulate what happens when 4 GPUs split a dataset.

**Your turn:**

1. Create 4 DistributedSamplers for the same dataset (rank 0-3, world_size=4)
2. Verify each sampler produces different indices
3. Verify the union of all indices covers the full dataset (no gaps, no duplicates)
4. Verify that `set_epoch()` changes the shuffle order

**Look up:** `DistributedSampler(dataset, num_replicas, rank, shuffle)`

In [ ]:
# EXERCISE: Simulate DistributedSampler across 4 GPUs

WORLD_SIZE = 4

# YOUR CODE:
# 1. Create a sampler for each rank
#    samplers = [DistributedSampler(dataset, num_replicas=WORLD_SIZE, rank=r, shuffle=True)
#               for r in range(WORLD_SIZE)]
samplers = ...

# 2. Set epoch to 0 for all samplers (deterministic for this test)
#    for s in samplers: s.set_epoch(0)
# YOUR CODE HERE

# 3. Collect indices from each sampler
#    indices_per_rank = [list(s) for s in samplers]
indices_per_rank = ...

# Print summary
for r in range(WORLD_SIZE):
    print(f"Rank {r}: {len(indices_per_rank[r])} samples, first 5: {indices_per_rank[r][:5]}")

# 4. Verify: no overlap between ranks
all_indices = []
for r in range(WORLD_SIZE):
    all_indices.extend(indices_per_rank[r])

print(f"\nTotal indices: {len(all_indices)}")
print(f"Unique indices: {len(set(all_indices))}")
print(f"Dataset size: {len(dataset)}")

# 5. Verify set_epoch changes the shuffle
# YOUR CODE:
# Set epoch to 1, get indices again for rank 0, compare with epoch 0
samplers[0].set_epoch(1)
epoch1_indices = list(samplers[0])
epoch0_indices = indices_per_rank[0]
same_order = epoch0_indices == epoch1_indices
print(f"\nSame order across epochs? {same_order} (should be False)")

In [ ]:
# VERIFY: DistributedSampler behavior

# Each rank should get roughly len(dataset) / world_size samples
expected_per_rank = len(dataset) // WORLD_SIZE
for r in range(WORLD_SIZE):
    actual = len(indices_per_rank[r])
    # DistributedSampler may pad to make evenly divisible
    assert actual >= expected_per_rank, \
        f"Rank {r}: expected >= {expected_per_rank}, got {actual}"

# All indices together should cover the dataset
# (DistributedSampler may duplicate a few samples to make even splits)
assert len(set(all_indices)) >= len(dataset) - WORLD_SIZE, \
    f"Not enough unique indices: {len(set(all_indices))} vs dataset {len(dataset)}"

# Ranks should have different indices (they're shuffled differently)
assert indices_per_rank[0] != indices_per_rank[1], "Rank 0 and 1 should differ"

# set_epoch should change order
assert not same_order, "set_epoch should change shuffle order"

print(f"Samples per rank: ~{expected_per_rank}")
print(f"Coverage: {len(set(all_indices))}/{len(dataset)} unique indices")
print("DistributedSampler: OK!")

## Part 8: torch.compile — Free Performance

`torch.compile` (PyTorch 2.0+) JIT-compiles your model into optimized kernels.
Typical speedup: 20-50% with zero code changes to the model.

```python
model = MyModel()
model = torch.compile(model)  # that's it
```

### Modes:
- `torch.compile(model, mode="default")` — balanced (good default)
- `torch.compile(model, mode="reduce-overhead")` — less compile time, CUDA graphs
- `torch.compile(model, mode="max-autotune")` — slower compile, fastest runtime

### What breaks torch.compile:
- Dynamic shapes (different batch sizes between calls)
- Data-dependent control flow (`if x.sum() > 0:`)
- In-place operations on tensors that are inputs to the graph
- Custom CUDA extensions (sometimes)

**Best practice for production:** compile individual submodules (e.g., the U-Net) rather than
the entire training loop. Use `torch._dynamo.config.suppress_errors = True` during development.

**Your turn:** Compile the SimpleCNN and compare inference speed.

In [ ]:
# EXERCISE: torch.compile benchmark
import time

model = SimpleCNN().to(device).eval()
test_input = torch.randn(16, 3, 64, 64, device=device)

# Warmup eager mode
with torch.no_grad():
    for _ in range(10):
        _ = model(test_input)
if device.type == "cuda":
    torch.cuda.synchronize()

# Benchmark eager
N_RUNS = 100
start = time.perf_counter()
with torch.no_grad():
    for _ in range(N_RUNS):
        _ = model(test_input)
if device.type == "cuda":
    torch.cuda.synchronize()
eager_time = (time.perf_counter() - start) / N_RUNS * 1000  # ms

# YOUR CODE:
# 1. Compile the model: compiled_model = torch.compile(model, mode="default")
# 2. Warmup compiled model (first call triggers compilation — will be slow)
#    with torch.no_grad(): _ = compiled_model(test_input)
#    if device.type == "cuda": torch.cuda.synchronize()
# 3. Benchmark compiled model same as above

compiled_model = ...

# Warmup (triggers JIT compilation)
# YOUR CODE
raise NotImplementedError

In [ ]:
# VERIFY: torch.compile results

# Benchmark compiled
start = time.perf_counter()
with torch.no_grad():
    for _ in range(N_RUNS):
        _ = compiled_model(test_input)
if device.type == "cuda":
    torch.cuda.synchronize()
compiled_time = (time.perf_counter() - start) / N_RUNS * 1000

# Verify outputs match
with torch.no_grad():
    eager_out = model(test_input)
    compiled_out = compiled_model(test_input)
    assert torch.allclose(eager_out, compiled_out, atol=1e-5), "Outputs should match"

speedup = eager_time / compiled_time if compiled_time > 0 else float("inf")
print(f"Eager:    {eager_time:.2f} ms/batch")
print(f"Compiled: {compiled_time:.2f} ms/batch")
print(f"Speedup:  {speedup:.2f}x")
print(f"\nNote: speedup varies by model size and GPU. Small models on CPU may not benefit.")
print("torch.compile: OK!")

## Recap: What You Now Have

| Part | Skill | Notebook pattern | Production pattern |
|------|-------|-----------------|--------------------|
| 1 | Tensor ops | Direct creation + indexing | Same — this is universal |
| 2 | Data pipeline | `Dataset` + `DataLoader(shuffle=True)` | `DistributedSampler` + WebDataset/MDS |
| 3 | Custom modules | `nn.Module` + `register_buffer` | Same + hooks for monitoring |
| 4 | Loss functions | Single `F.mse_loss` | Multi-component weighted loss |
| 5 | Training loop | Manual loop + checkpoint | + LR warmup + grad clipping + resume |
| 6 | Performance | `torch.amp.autocast` | + gradient accumulation |
| 7 | Distributed | N/A | DDP → FSDP → multi-node `torchrun` |
| 8 | Compilation | `torch.compile` | `mode="max-autotune"` per submodule |

### What to do next:

→ **Notebook 01** uses everything from Parts 1-5: you'll build a Dataset, write transforms,
construct a U-Net from `nn.Module` blocks, implement a training loop, and run sampling.

→ Parts 6-8 become relevant starting from Notebook 07 (inference optimization) and
Notebook 19 (distributed training infrastructure).

### Quick reference — patterns you'll use constantly:

```python
# Index a 1D schedule with batch of timesteps, reshape for broadcasting
vals = schedule[t].view(-1, 1, 1, 1)  # (B,) -> (B, 1, 1, 1)

# Register non-trainable tensors that move with the model
self.register_buffer("alpha_bar", torch.cumprod(1 - betas, dim=0))

# Conditioning injection: project vector to channels, broadcast over spatial
cond_bias = self.proj(cond_vec)[:, :, None, None]  # (B, C, 1, 1)
h = h + cond_bias

# AMP training step
with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
    loss = model(x)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

# Checkpoint with full resume
torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict(), "epoch": epoch}, path)
```